# Fine-tune Llama 3.2 3B on your own data (Unsloth, Colab)

Runtime → Change runtime type → T4 GPU. Upload `data/processed/train.jsonl` and `val.jsonl` from `python -m own_data_sft prepare`.

In [ ]:
!pip install -q unsloth

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

SYSTEM = "You are the assistant for a small pottery studio. Answer briefly and only with facts you know."

model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Llama-3.2-3B-Instruct", max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

In [ ]:
def to_text(batch):
    out = []
    for msgs in batch["messages"]:
        if msgs[0]["role"] != "system":
            msgs = [{"role": "system", "content": SYSTEM}] + msgs
        out.append(tokenizer.apply_chat_template(msgs, tokenize=False))
    return {"text": out}

ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})
ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)

In [ ]:
trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds["train"], eval_dataset=ds["val"],
    args=SFTConfig(dataset_text_field="text", max_seq_length=1024, per_device_train_batch_size=4,
        gradient_accumulation_steps=2, num_train_epochs=3, learning_rate=2e-4, warmup_steps=5,
        logging_steps=5, optim="adamw_8bit", lr_scheduler_type="linear", seed=3407,
        output_dir="outputs/own-data-3b", report_to="none"))
trainer = train_on_responses_only(trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")
print("val loss before", trainer.evaluate()["eval_loss"])
trainer.train()
print("val loss after", trainer.evaluate()["eval_loss"])

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "How much is the beginner wheel course?"}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(tokenizer.decode(model.generate(input_ids=ids, max_new_tokens=120, do_sample=False)[0, ids.shape[1]:], skip_special_tokens=True))
model.save_pretrained("outputs/own-data-3b/adapter")
tokenizer.save_pretrained("outputs/own-data-3b/adapter")